In [ ]:
! pip install numpyro healpy einops reproject

In [4]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
import os
os.environ["XLA_FLAGS"] = "--xla_gpu_force_compilation_parallelism=1"

from google.colab import drive
drive.mount('/content/drive')
sys.path.append(r'/content/drive/Othercomputers/My MacBook Pro/fermi-prob-prog/')
%cd /content/drive/Othercomputers/My MacBook Pro/fermi-prob-prog/notebooks

import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

import numpy as np
import arviz as az
import healpy as hp
import pickle
from tqdm import tqdm

print(jax.devices())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/Othercomputers/My MacBook Pro/fermi-prob-prog/notebooks
[gpu(id=0)]


In [5]:
save_dir = "../outputs/poisson_sim/run_230718"
from models.np_model import NPModel
npmodel = NPModel(
    non_poissonian=True,
    l_max=2,
    vary_gamma=True,
    bulge_hybrid=True,
    ps_cat="3fgl",
    nside=128,
)

Loading the psf correction from: /content/drive/Othercomputers/My MacBook Pro/fermi-prob-prog/notebooks/psf_dir/Fermi_PSF_2GeV2_nside128.npy
Max photon count is 103


In [ ]:
for i in range(1,2):
    counts = jnp.array(np.load(f"{save_dir}/counts_{i}.npy"), dtype=jnp.int32)
    svi_results = npmodel.fit_svi(
        rng_key=jax.random.PRNGKey(4234),
        n_steps=2000,
        guide="iaf",
        lr=5e-5,
        num_particles=8,
        data=jnp.array(counts),
    )
    pickle.dump(svi_results, open(f"{save_dir}/svi_results_{i}.p", 'wb'))
    samples = npmodel.get_svi_samples(
        rng_key=jax.random.PRNGKey(42),
        num_samples=50000,
    )
    pickle.dump(samples, open(f"{save_dir}/samples_{i}.p", 'wb'))

100%|██████████| 2000/2000 [09:40<00:00,  3.45it/s, init loss: 24799.2821, avg. loss [1901-2000]: 18696.0463]
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-9-b09f9504cc1c>", line 11, in <cell line: 1>
    pickle.dump(svi_results, open(f"{save_dir}/svi_results_{i}.p", 'wb'))
OSError: [Errno 107] Transport endpoint is not connected: '../outputs/poisson_sim/run_230718/svi_results_1.p'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
AttributeError: 'OSError' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/ultratb.py", line 1101, in get_records
    return _fixed_getinn